# 03 · Cancellation model: which bookings will be cancelled?

**Question:** can the risk of cancellation be estimated on the day a booking is made?

**Rules:**
- Only information **known at booking time** is used (no data leakage).
- The model learns from arrivals in **2015–2016** and is tested on arrivals in **2017**, as it would be used in practice.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

df = pd.read_csv("../data/processed/hotel_bookings_clean.csv", parse_dates=["arrival_date"])
print(df.shape)

## 3.1 One more leakage check: parking spaces
`required_car_parking_spaces` looks harmless, but when is it recorded?

In [ ]:
(df.groupby(df["required_car_parking_spaces"] > 0)["is_canceled"].mean() * 100).round(1)

Bookings with a parking space are **never** cancelled (0.0% vs 39.5%). This is not credible behaviour: the space is most likely recorded at arrival, so the column reveals that the guest showed up. It is excluded, like `reservation_status`.

## 3.2 Features

Excluded on purpose:
- **Known only after booking:** `reservation_status`, `reservation_status_date`, `assigned_room_type`, `booking_changes`, `required_car_parking_spaces`
- **Built from the outcome:** `is_duplicate` (computed on all columns, including `is_canceled`)
- **Redundant or not useful:** `arrival_date`, `arrival_date_year`, `revenue`, `agent` (an ID with hundreds of values)

In [ ]:
top_countries = df["country"].value_counts().index[:20]
df["country_grp"] = df["country"].where(df["country"].isin(top_countries), "Other")

NUM = ["lead_time", "arrival_date_week_number", "stays_in_weekend_nights", "stays_in_week_nights",
       "adults", "children", "babies", "is_repeated_guest", "previous_cancellations",
       "previous_bookings_not_canceled", "days_in_waiting_list", "adr",
       "total_of_special_requests", "has_company"]
CAT = ["hotel", "arrival_date_month", "meal", "country_grp", "market_segment",
       "distribution_channel", "reserved_room_type", "deposit_type", "customer_type"]
TARGET = "is_canceled"

## 3.3 Time-based split

In [ ]:
train = df[df["arrival_date"] < "2017-01-01"]
test = df[df["arrival_date"] >= "2017-01-01"]
y_train, y_test = train[TARGET], test[TARGET]

print(f"Train: {len(train):,} bookings, cancellation rate {y_train.mean():.1%}")
print(f"Test:  {len(test):,} bookings, cancellation rate {y_test.mean():.1%}")

Train: 78,589 arrivals in 2015–2016 (cancellation rate 36.2%). Test: 40,619 arrivals in January–August 2017 (38.7%). The two rates are close, so the test set reflects the same problem.

## 3.4 Baselines
A model is only useful if it beats simple rules a hotel could already apply.

In [ ]:
def evaluate(name, y_true, y_pred, y_score=None):
    """Main metrics of a prediction as one table row."""
    return {
        "model": name,
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 3),
        "recall": round(recall_score(y_true, y_pred), 3),
        "roc_auc": round(roc_auc_score(y_true, y_score), 3) if y_score is not None else None,
    }

results = [evaluate("Always 'not cancelled'", y_test, np.zeros(len(y_test), dtype=int))]
rule = ((test["lead_time"] > 90) | (test["deposit_type"] == "Non Refund")).astype(int)
results.append(evaluate("Rule: lead time > 90 or Non Refund", y_test, rule))
pd.DataFrame(results)

- Always predicting "not cancelled" is right 61% of the time but finds **no** cancellations: accuracy alone is misleading.
- The rule finds two thirds of cancellations (recall 66.9%), but only half of the flagged bookings are actually cancelled (precision 51.8%).

## 3.5 Models

Preparation and model are combined in a pipeline, fitted on the training data only:
- **Logistic regression:** simple and explainable
- **Random forest:** many decision trees voting together, able to capture non-linear patterns

In [ ]:
def make_pipeline(num, cat, model):
    prep = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])
    return Pipeline([("prep", prep), ("model", model)])

def fit_and_score(name, num, cat, model):
    pipe = make_pipeline(num, cat, model)
    pipe.fit(train[num + cat], y_train)
    score = pipe.predict_proba(test[num + cat])[:, 1]
    results.append(evaluate(name, y_test, (score >= 0.5).astype(int), score))
    return pipe, score

logreg, _ = fit_and_score("Logistic regression", NUM, CAT, LogisticRegression(max_iter=1000))
rf_all, _ = fit_and_score("Random forest", NUM, CAT, RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=42))
pd.DataFrame(results)

- Both models clearly beat the rule on precision (**~80% vs 52%**) and accuracy (**77% vs 63%**), with ROC AUC **0.846** (random forest) and **0.839** (logistic regression): the simple model is almost as good.
- At the default 0.5 threshold, recall (54%) is lower than the rule's (67%): the model only flags bookings it is confident about. The threshold is a business choice, addressed in notebook 04.

## 3.6 What drives the model, and can every feature be trusted?
**Permutation importance:** each column is shuffled in the test set and the drop in ROC AUC is measured. A large drop means the model relies on that column.

In [ ]:
def importance_of(pipe, features, n=5000):
    sample = test[features].sample(min(n, len(test)), random_state=42)
    imp = permutation_importance(pipe, sample, y_test.loc[sample.index], scoring="roc_auc",
                                 n_repeats=5, random_state=42, n_jobs=-1)
    return pd.Series(imp.importances_mean, index=features).sort_values(ascending=False)

importance_of(rf_all, NUM + CAT).head(10).round(3)

`country` is the most important feature. Since nationality is often recorded or corrected at check-in (from the guest's ID), it may partly be information from the future. Two checks:

In [ ]:
top8 = df["country"].value_counts().index[:8]
c = df[df["country"].isin(top8)].groupby("country").agg(
    n_bookings=("is_canceled", "size"),
    cancel_rate=("is_canceled", "mean"),
)
c["cancel_rate"] = (c["cancel_rate"] * 100).round(1)
c.sort_values("n_bookings", ascending=False)

In [ ]:
CAT_NO_DEP = [col for col in CAT if col != "deposit_type"]
CAT_NO_COUNTRY = [col for col in CAT if col != "country_grp"]

rf_no_dep, _ = fit_and_score("Random forest (no deposit_type)", NUM, CAT_NO_DEP, RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=42))
final, final_score = fit_and_score("Random forest (no country) - final", NUM, CAT_NO_COUNTRY, RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=42))
pd.DataFrame(results)

- **Portugal** accounts for 48,482 bookings (41%) with a cancellation rate of **56.7%**, vs 17–35% for other countries. This can be real behaviour (domestic guests cancel more easily; many group blocks are Portuguese) or a recording effect (country set by default at booking and corrected at check-in, which cancelled bookings never reach). The data cannot separate the two.
- Without `country`, ROC AUC drops from 0.846 to **0.824**: a moderate cost.
- Without `deposit_type`, it barely changes (0.840): the same group blocks are also captured by segment, lead time and country.

**Decision:** the final model excludes `country`. A model used on the day of booking must not depend on information that may be recorded at check-in: a slightly weaker model that can be trusted is preferred.

## 3.7 Final model

In [ ]:
FINAL = NUM + CAT_NO_COUNTRY
cm = confusion_matrix(y_test, (final_score >= 0.5).astype(int))
display(pd.DataFrame(cm, index=["Actual: stayed", "Actual: cancelled"],
                     columns=["Predicted: stay", "Predicted: cancel"]))

importance = importance_of(final, FINAL).head(10)
fig, ax = plt.subplots(figsize=(8, 4.5))
imp_sorted = importance.sort_values()
ax.barh(imp_sorted.index, imp_sorted.values, color="#2a78d6", height=0.7)
for i, v in enumerate(imp_sorted.values):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9, color="#444")
ax.xaxis.set_visible(False)
for side in ["top", "right", "bottom"]:
    ax.spines[side].set_visible(False)
ax.spines["left"].set_color("#ccc")
ax.tick_params(axis="y", labelsize=9, length=0)
ax.set_title("Top 10 features of the final model (drop in ROC AUC when shuffled)", loc="left", fontsize=12)
plt.tight_layout()
plt.savefig("../figures/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

The top five features are deposit type, special requests, lead time, market segment and customer type: the same signals found in the exploratory analysis (notebook 02). Beyond them, importance drops below 0.01, so the model relies on a few clear signals.

In [ ]:
out = test[["hotel", "arrival_date", "lead_time", "deposit_type", "market_segment",
            "adr", "total_nights", "revenue", "is_canceled"]].copy()
out["cancel_score"] = final_score
out.to_csv("../data/processed/test_predictions.csv", index=False)
print(out.shape)

## Findings

1. **Leakage removed before modelling.** `reservation_status`, `reservation_status_date`, `assigned_room_type` and `booking_changes` are set after booking; `required_car_parking_spaces` is also leakage (0.0% cancellation when a space is recorded).
2. **The model clearly beats simple rules.** Trained on 2015–2016 and tested on 2017, it flags cancellations with ~80% precision vs 52% for a "lead time > 90 days or Non Refund" rule, with ROC AUC up to 0.85. Logistic regression is almost as good (0.84).
3. **A slightly weaker model that can be trusted.** `country` was the top feature (guests from Portugal cancel 56.7% vs 17–35%), but nationality may be recorded at check-in: the final model excludes it (ROC AUC 0.846 → 0.824).
4. **The model relies on the same signals found in the analysis:** deposit type, special requests, lead time, market segment and customer type.

**Final model:** random forest without `country`: accuracy 76.1%, precision 78.4%, recall 52.9%, ROC AUC 0.824.

**Use:** the model is not meant to decide on single bookings. It ranks bookings by risk and estimates expected cancellations, to prioritise actions such as reconfirmation requests (notebook 04).